In [1]:
pip install tmu

In [2]:
from tmu.preprocessing.standard_binarizer.binarizer import StandardBinarizer
from tmu.models.regression.vanilla_regressor import TMRegressor
import numpy as np
import pandas as pd  # to read dataset_1_Train,and other such csv files
from sklearn import datasets
from sklearn.model_selection import train_test_split
import logging
import argparse
from tmu.tools import BenchmarkTimer

_LOGGER = logging.getLogger(__name__)

ERROR:tmu.clause_bank.clause_bank_cuda:No module named 'pycuda'
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/tmu/clause_bank/clause_bank_cuda.py", line 41, in <module>
    from pycuda._driver import Device, Context
ModuleNotFoundError: No module named 'pycuda'


In [3]:
def metrics(args):
    return dict(
        rmsd=[],
        train_time=[],
        test_time=[]
    )

In [6]:
def main(args):
    experiment_results = metrics(args)

    # california_housing = datasets.fetch_california_housing()
    # X = california_housing.data
    # Y = california_housing.target

    df = pd.read_csv("dataset1_train.csv")
    X_train = df.iloc[:, :-1].values  # for all the columns except last
    Y_train = df.iloc[:, -1].values   # for the last column
    print(X_train.dtype)
    print(Y_train.dtype)

    df1 = pd.read_csv("dataset1_test.csv")
    X_test = df1.iloc[:, :-1].values  # for all the columns except last
    Y_test = df1.iloc[:, -1].values   # for the last column


    # b = StandardBinarizer(max_bits_per_feature=10)
    # X_train_transformed = b.fit_transform(X_train)
    # X_test_transformed = b.transform(X_test)

    #X_transformed = np.concatenate((X_train_transformed, X_test_transformed), axis=0

    tm = TMRegressor(
        args.num_clauses,
        args.T,
        args.s,
        platform=args.platform,
        weighted_clauses=args.weighted_clauses
    )

    tm_results = np.empty(0)

    _LOGGER.info(f"Running RMSD with {TMRegressor} for {args.num_runs}")
    for run in range(args.num_runs):
        # X_train, X_test, Y_train, Y_test = train_test_split(X_transformed, Y)

        for epoch in range(args.epochs):
            benchmark1 = BenchmarkTimer(logger=_LOGGER, text="Training Time")
            with benchmark1:
                tm.fit(X_train, Y_train)
            experiment_results["train_time"].append(benchmark1.elapsed())

            benchmark2 = BenchmarkTimer(logger=_LOGGER, text="Testing Time")
            with benchmark2:
                tm_results = np.append(tm_results, np.sqrt(((tm.predict(X_test) - Y_test) ** 2).mean()))
            experiment_results["test_time"].append(benchmark1.elapsed())
            experiment_results["rmsd"].append(tm_results.mean())

            _LOGGER.info(
                f"#{run + 1} - Epoch: {epoch}: RMSD: {tm_results.mean()} +/- {1.96 * tm_results.std() / np.sqrt(run + 1)} ({benchmark1.elapsed()})")


    # to extract weights and clauses
    #weights = tm.weight_bank.get_weights()
    clauses = tm.clause_bank.get_literals().astype(np.uint8)

    print("Clauses shape:", clauses.shape)
    #print("Weights shape:", weights.shape)

    # clause file in .mem format
    with open("clauses.mem", "w") as f:
         for clause in clauses:
             f.write(" ".join(map(str, clause)))
             f.write("\n")

    # for Xtest_mem file which has all the literals
    with open("Xtest_6.mem", "w") as f:
        for row in X_test.astype(np.uint8):
             pos = row
             neg = 1 - row
             expanded = np.concatenate([pos, neg])
             f.write("".join(map(str, expanded)) + "\n")

    # for Y_test_mem file
    with open("Y_test.mem", "w") as f:
        for y in Y_test:
            f.write(f"{int(y):032b}\n")


    Y_pred = tm.predict(X_test)

    abs_error = 0

    for i in range(100):       # only first 100 samples for verifying with hardware
        diff = Y_pred[i] - Y_test[i]
        print(f"Output={Y_pred[i]} Ytest={Y_test[i]} diff={diff}")
        abs_error += abs(diff)

    print("MAE =", abs_error / 100.0)
        # mae = np.mean(np.abs(Y_pred - Y_test))
        # print("MAE =", mae)


In [7]:
def default_args(**kwargs):
    parser = argparse.ArgumentParser()
    parser.add_argument("--num_clauses", default=7, type=int) #given in paper
    parser.add_argument("--T", default=7, type=int)
    parser.add_argument("--s", default=2, type=int) #given in paper
    parser.add_argument("--platform", default="CPU", type=str)
    parser.add_argument("--weighted_clauses", default=False, type=bool) #not weighted
    parser.add_argument("--epochs", default=196, type=int) # according to paper
    parser.add_argument("--num-runs", default=3, type=int) # for this example you dont need 25 even 1 is fine
    args, _ = parser.parse_known_args()
    for key, value in kwargs.items():
        if key in args.__dict__:
            setattr(args, key, value)
    return args


if __name__ == "__main__":
    results = main(default_args())
    _LOGGER.info(results)

int64
int64
Clauses shape: (7, 6)
Output=700.0 Ytest=700 diff=0.0
Output=200.0 Ytest=200 diff=0.0
Output=200.0 Ytest=200 diff=0.0
Output=0.0 Ytest=0 diff=0.0
Output=400.0 Ytest=400 diff=0.0
Output=500.0 Ytest=500 diff=0.0
Output=100.0 Ytest=100 diff=0.0
Output=400.0 Ytest=400 diff=0.0
Output=700.0 Ytest=700 diff=0.0
Output=200.0 Ytest=200 diff=0.0
Output=0.0 Ytest=0 diff=0.0
Output=500.0 Ytest=500 diff=0.0
Output=700.0 Ytest=700 diff=0.0
Output=200.0 Ytest=200 diff=0.0
Output=100.0 Ytest=100 diff=0.0
Output=200.0 Ytest=200 diff=0.0
Output=700.0 Ytest=700 diff=0.0
Output=700.0 Ytest=700 diff=0.0
Output=100.0 Ytest=100 diff=0.0
Output=300.0 Ytest=300 diff=0.0
Output=400.0 Ytest=400 diff=0.0
Output=500.0 Ytest=500 diff=0.0
Output=0.0 Ytest=0 diff=0.0
Output=300.0 Ytest=300 diff=0.0
Output=200.0 Ytest=200 diff=0.0
Output=700.0 Ytest=700 diff=0.0
Output=600.0 Ytest=600 diff=0.0
Output=300.0 Ytest=300 diff=0.0
Output=600.0 Ytest=600 diff=0.0
Output=200.0 Ytest=200 diff=0.0
Output=200.0 Ytest

In [7]:
!sed -i 's/np.uint32(~0)/np.uint32(0xFFFFFFFF)/g' /usr/local/lib/python3.12/dist-packages/tmu/clause_bank/clause_bank.py :

sed: can't read :: No such file or directory


In [8]:
!grep -n "uint32" /usr/local/lib/python3.12/dist-packages/tmu/clause_bank/clause_bank.py | head :

head: cannot open ':' for reading: No such file or directory


In [9]:
!sed -n '130,140p' /usr/local/lib/python3.12/dist-packages/tmu/clause_bank/clause_bank.py :

        self.clause_bank = np.empty(
            shape=(self.number_of_clauses, self.number_of_ta_chunks, self.number_of_state_bits_ta),
            dtype=np.uint32,
            order="c"
        )

        self.clause_bank[:, :, 0: self.number_of_state_bits_ta - 1] = np.uint32(0xFFFFFFFF)
        self.clause_bank[:, :, self.number_of_state_bits_ta - 1] = 0
        self.clause_bank = np.ascontiguousarray(self.clause_bank.reshape(
            (self.number_of_clauses * self.number_of_ta_chunks * self.number_of_state_bits_ta)))

sed: can't read :: No such file or directory


In [4]:
from google.colab import files
uploaded = files.upload()

Saving dataset1_test.csv to dataset1_test (5).csv


In [5]:
from google.colab import files
uploaded = files.upload()

Saving dataset1_train.csv to dataset1_train (5).csv
